# Chemical-gas humidity analysis

This notebook is fixed to the chemical-gas humidity experiment. It computes
directional dSNR, mutual information (MI), and true-class SHAP from a saved
training run. SHAP uses the model-training partition as background and only the
external heldout-test chips for evaluation.


In [ ]:
# Use the included humid model, or enter your own saved run folder.
RUN_FOLDER_NAME = "pretrained"
# False displays the verified saved SHAP; True recomputes GradientSHAP.
RECOMPUTE_SHAP = False

from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ML_models
ML_models = importlib.reload(ML_models)
from configs import load_config
from src import Analysis_tools, Data_pipeline, Metric_preprocessing, Run_analysis, Shap_reference

Analysis_tools = importlib.reload(Analysis_tools)
Metric_preprocessing = importlib.reload(Metric_preprocessing)
Shap_reference = importlib.reload(Shap_reference)
Data_pipeline = importlib.reload(Data_pipeline)
Run_analysis = importlib.reload(Run_analysis)
PAPER_CONFIG = load_config("Config_TBDT_Chem_humid_pretrained")
ANALYSIS = PAPER_CONFIG["ANALYSIS"]
OUTPUT = PAPER_CONFIG["OUTPUT"]

if not RUN_FOLDER_NAME.strip() or RUN_FOLDER_NAME.strip().lower() == "latest":
    raise ValueError("Set RUN_FOLDER_NAME to the exact saved training-run folder name.")
runs_root = PROJECT_ROOT / OUTPUT["models_dir"] / OUTPUT["runs_dirname"]
run_dir = Run_analysis.resolve_run_dir(RUN_FOLDER_NAME, runs_root)
run_metadata = Run_analysis.load_run_metadata(run_dir)
RUN_CONFIG = run_metadata["config"]
if RUN_CONFIG.get("CONFIG_NAME") != "Config_TBDT_Chem_humid_pretrained":
    raise ValueError("The selected run is not a Config_TBDT_Chem_humid_pretrained run.")
if RUN_CONFIG["MODEL"].get("model_name") != "CNN1D_TBDT":
    raise ValueError("The selected run does not use CNN1D_TBDT.")

DATASET = RUN_CONFIG["DATASET"]
PREPROCESS = RUN_CONFIG["PREPROCESS"]
SELECTION = RUN_CONFIG["SELECTION"]
DATA_ROOT = Data_pipeline.resolve_data_root(DATASET["data_dir"])
train_rh_values = {float(value) for value in SELECTION["train_rh_values"]}
heldout_rh_values = {float(value) for value in SELECTION["heldout_test_rh_values"]}
if train_rh_values != {0.0, 50.0} or heldout_rh_values != {0.0, 30.0, 50.0}:
    raise ValueError(
        "Expected training RH [0, 50] and heldout-test RH [0, 30, 50] "
        "for the chemical-gas humidity analysis."
    )

print("Selected run folder:", RUN_FOLDER_NAME)
print("SHAP background: training chips, model-training partition, RH 0/50")
print("SHAP evaluation: every heldout-test RH 30 sample (7,572 in the published dataset)")


## 1. Load the data and check raw-current quality

MI and dSNR use all RH 0/50 observations from the training and held-out folders
that pass the raw-current quality checks across every selected Vg channel.
Both metrics use the same cohort, with no random sample limit. The published
dataset contains 44,312 such observations. Model and SHAP inputs remain separate.


In [ ]:
analysis_bundle = Data_pipeline.prepare_data_bundle(
    data_root=DATA_ROOT,
    train_rh_values=SELECTION["train_rh_values"],
    heldout_test_rh_values=SELECTION["heldout_test_rh_values"],
    average_mode=PREPROCESS["average_mode"],
    cut_window=PREPROCESS["cut_window"],
    val_size=SELECTION["train_val_split"]["val_size"],
    random_state=SELECTION["train_val_split"]["random_state"],
)
if analysis_bundle["gas_label_map"] != run_metadata["gas_label_map"]:
    raise ValueError("Current data labels do not match the selected run.")
display(Data_pipeline.sample_summary(analysis_bundle))

metric_data = Metric_preprocessing.prepare_metric_data(
    analysis_bundle, data_root=DATA_ROOT,
    average_mode=PREPROCESS["average_mode"],
    rh_values=ANALYSIS["metric_rh_values"],
    **ANALYSIS["metric_quality"],
)
mi_data = dsnr_data = metric_data
print("MI and dSNR samples after raw-current QC:", len(metric_data["y"]))
display(metric_data["group_summary"])


## 2. Directional dSNR

For every gas pair and gate voltage, residual variation is projected onto the
axis connecting the two gas centroids. Pair separation is divided by the
pooled projected standard deviation, and the reported dSNR is the geometric
mean across all gas pairs.


In [ ]:
dsnr_result = Analysis_tools.compute_directional_dsnr(
    dsnr_data["x"],
    dsnr_data["y"],
    analysis_bundle["vg_axis"],
)
dsnr_result["sample_metadata"] = dsnr_data["metadata"]
display(dsnr_result["summary"].head())
dsnr_fig, dsnr_ax = Analysis_tools.plot_dsnr(dsnr_result)
plt.show()


## 3. Mutual information

The heatmap uses every scalar time/Vg coordinate. The Vg profile uses joint
vectors of ten selected time points; time-wise MI uses six selected Vg points.
Both use all 44,312 quality-filtered RH 0/50 observations, k = 5, and seed 0.
Indices below are zero-based within the 42-time-point by 96-Vg-point analysis
window. The time-wise results are saved numerically without adding a plot.


In [ ]:
mi_result = Analysis_tools.compute_mutual_information(
    mi_data["x"],
    mi_data["y"],
    joint_time_indices=ANALYSIS["mi_joint_time_indices"],
    joint_vg_indices=ANALYSIS["mi_joint_vg_indices"],
    neighbors=ANALYSIS["mi_neighbors"],
    random_state=ANALYSIS["random_seed"],
)
mi_result.update({key: metric_data[key] for key in (
    "quality_summary", "sample_qc", "channel_qc", "group_summary")})
mi_result["sample_metadata"] = metric_data["metadata"]
mi_result["time_coordinates"] = pd.DataFrame({
    "time_index": mi_result["joint_time_indices"],
    "raw_time_index": mi_result["joint_time_indices"] + PREPROCESS["cut_window"]["t_start"],
    "time_min": analysis_bundle["time_axis"][mi_result["joint_time_indices"]],
})
mi_result["vg_coordinates"] = pd.DataFrame({
    "vg_index": mi_result["joint_vg_indices"],
    "aligned_vg_index": mi_result["joint_vg_indices"] + PREPROCESS["cut_window"]["v_start"],
    "vg_volts": analysis_bundle["vg_axis"][mi_result["joint_vg_indices"]],
})
print("Time points used for MI versus Vg:")
display(mi_result["time_coordinates"])
print("Vg points used for MI versus time:")
display(mi_result["vg_coordinates"])
mi_fig, mi_axes = Analysis_tools.plot_mutual_information(
    mi_result,
    analysis_bundle["time_axis"],
    analysis_bundle["vg_axis"],
)
plt.show()


## 4. SHAP for held-out RH 30

The reference background is the archived 8,192 model-training observations at
RH 0/50; evaluation uses all 7,572 held-out RH 30 observations. Attributions
explain the true gas logit over all 42 × 96 input coordinates. Absolute values
are averaged within each gas, then the ten gas means receive equal weight.
The displayed profiles average 8 adjacent Vg points or 2 consecutive time points.
Only the time plot omits negative-time stabilization points.

By default, the verified saved maps display the archived curves without
recomputing SHAP. Set `RECOMPUTE_SHAP = True` to run GradientSHAP with 200
integration samples and the same observations. Recomputed estimates can differ
because the archived integration seed was not recorded. No model is trained.


In [ ]:
shap_device = ANALYSIS["shap_device"]
model_device = "cuda" if shap_device == "auto" and __import__("torch").cuda.is_available() else (
    "cpu" if shap_device == "auto" else shap_device
)
model, loaded_model_metadata = Run_analysis.load_model(run_dir, device=model_device)
reference_dir = PROJECT_ROOT / ANALYSIS["shap_reference_dir"]
if RECOMPUTE_SHAP:
    shap_result = Analysis_tools.compute_shap(
        model, analysis_bundle,
        background_size=ANALYSIS["shap_background_size"],
        eval_size=ANALYSIS["shap_eval_size"],
        background_rh_values=ANALYSIS["shap_background_rh_values"],
        evaluation_rh_values=ANALYSIS["shap_evaluation_rh_values"],
        batch_size=ANALYSIS["shap_batch_size"],
        nsamples=ANALYSIS["shap_nsamples"],
        vg_bin_size=ANALYSIS["shap_vg_bin_size"],
        time_bin_size=ANALYSIS["shap_time_bin_size"],
        device=model_device, random_state=ANALYSIS["random_seed"],
        reference_dir=reference_dir,
    )
else:
    shap_result = Shap_reference.load_reference_shap(
        model, analysis_bundle, reference_dir,
        vg_bin_size=ANALYSIS["shap_vg_bin_size"],
        time_bin_size=ANALYSIS["shap_time_bin_size"],
    )
display(shap_result["background_metadata"].groupby(["split", "model_split", "rh_percent", "gas"])
        .size().rename("background_samples").reset_index())
display(shap_result["eval_metadata"].groupby(["split", "rh_percent", "gas"])
        .size().rename("evaluated_samples").reset_index())
print("SHAP source:", shap_result["source"])
shap_fig, shap_axes = Analysis_tools.plot_shap_profiles(
    shap_result, yscale=ANALYSIS["shap_yscale"],
)
plt.show()


## 5. Save numerical results and figures


In [ ]:
analysis_output_dir = PROJECT_ROOT / "results" / "analysis" / "chemical_gas_all_humidity" / RUN_FOLDER_NAME
figure_output_dir = PROJECT_ROOT / "figures" / "analysis" / "chemical_gas_all_humidity" / RUN_FOLDER_NAME
Analysis_tools.save_analysis_results(
    analysis_output_dir,
    dsnr_result=dsnr_result,
    mi_result=mi_result,
    shap_result=shap_result,
)
figure_output_dir.mkdir(parents=True, exist_ok=True)
dsnr_fig.savefig(figure_output_dir / "dsnr_by_vg.png", dpi=300, bbox_inches="tight")
mi_fig.savefig(figure_output_dir / "mutual_information.png", dpi=300, bbox_inches="tight")
shap_fig.savefig(figure_output_dir / "shap_RH30_vg_time.png", dpi=300, bbox_inches="tight")
print("Numerical results:", analysis_output_dir)
print("Figures:", figure_output_dir)
